In [ ]:
import os 
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

In [ ]:
import sys
sys.path.append('../')

In [ ]:
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
import seaborn as sns
import torch

In [ ]:
import pickle
import random
import h5py

from torch.utils.data import IterableDataset
from torch_geometric.data import Data, DataLoader

from Utils import AU2EV, rmsd_loss, d_mae_loss, Kabsch_alignment, pairwise_dist_to_coord, generate_fully_connected, count_negative_eig

In [ ]:
import torch_geometric

In [ ]:
REFERENCE_ENERGIES = {
    1: -13.62222753701504,
    6: -1029.4130839658328,
    7: -1484.8710358098756,
    8: -2041.8396277138045,
    9: -2712.8213146878606,
}

In [ ]:
def get_molecular_reference_energy(atomic_numbers):
    molecular_reference_energy = 0
    for atomic_number in atomic_numbers:
        molecular_reference_energy += REFERENCE_ENERGIES[atomic_number]

    return molecular_reference_energy

def generator(formula, rxn, grp):
    """ Iterates through a h5 group """

    energies = grp["wB97x_6-31G(d).energy"]
    forces = grp["wB97x_6-31G(d).forces"]
    atomic_numbers = list(grp["atomic_numbers"])
    positions = grp["positions"]
    molecular_reference_energy = get_molecular_reference_energy(atomic_numbers)

    for energy, force, positions in zip(energies, forces, positions):
        d = {
            "rxn": rxn,
            "wB97x_6-31G(d).energy": energy.__float__(),
            "wB97x_6-31G(d).atomization_energy": energy
            - molecular_reference_energy.__float__(),
            "wB97x_6-31G(d).forces": force.tolist(),
            "positions": positions,
            "formula": formula,
            "atomic_numbers": atomic_numbers,
        }

        yield d

def get_dynamics_data(formula, rxn, data):
    reactant = next(generator(formula, rxn, data[formula][rxn]["reactant"]))
    product = next(generator(formula, rxn, data[formula][rxn]["product"]))
    transition_state = next(generator(formula, rxn, data[formula][rxn]["transition_state"]))
    x = torch.tensor(reactant['atomic_numbers'], dtype=torch.long)

    reactant_pos = torch.tensor(reactant['positions'], dtype=torch.float32)
    product_pos = torch.tensor(product['positions'], dtype=torch.float32)
    transition_state_pos = torch.tensor(transition_state['positions'], dtype=torch.float32)

    energies = list()
    energies.append(torch.tensor(reactant['wB97x_6-31G(d).energy'], dtype=torch.float32))
    energies.append(torch.tensor(product['wB97x_6-31G(d).energy'], dtype=torch.float32))
    energies.append(torch.tensor(transition_state['wB97x_6-31G(d).energy'], dtype=torch.float32))
    
    return Data(x=x, reactant_pos=reactant_pos, product_pos=product_pos, transition_state_pos=transition_state_pos, energies=torch.stack(energies))

class Dataset_dynamics(IterableDataset):
    def __init__(self, hdf5_file, datasplit):
        super(Dataset_dynamics, self).__init__()
        self.hdf5_file = hdf5_file
        self.datasplit = datasplit
        assert datasplit in [
            "train",
            "valid",
            "test",
        ]
        with open('../Data/reactions_'+self.datasplit+'.pickle', 'rb') as f:
            self.datalist = pickle.load(f)

    def __iter__(self):
        with h5py.File(self.hdf5_file, "r") as f:
            data = f['data']
            i = 0
            if self.datasplit == 'train':
                random.shuffle(self.datalist)
            for formula, rxn in self.datalist:
                yield get_dynamics_data(formula, rxn, data)
                    
    def __len__(self):
        pass
    
def generate_dataloader_dynamics(hdf5_file, batch_size):
    dataloaders = {}
    dataloaders['train'] = DataLoader(dataset=Dataset_dynamics(hdf5_file, 'train'), batch_size = batch_size)
    dataloaders['val'] = DataLoader(dataset=Dataset_dynamics(hdf5_file, 'valid'), batch_size = batch_size)
    dataloaders['test'] = DataLoader(dataset=Dataset_dynamics(hdf5_file, 'test'), batch_size = batch_size)
    return dataloaders

In [ ]:
dataloader = generate_dataloader_dynamics('Transition1x/transition1x.h5', 1)

In [ ]:
def get_positions_from_pyscf(pyscf_dft_res):
    atom_type_and_pos = pyscf_dft_res[0].mol.atom
    atom_pos = [temp[1] for temp in atom_type_and_pos]
    return np.vstack(atom_pos)

In [ ]:
def reaction_stats(true_pos_list, pred_pos_list, energy_diff_list, dft_res_list, success_list=None):
    rmsds = []
    dmaes = []
    energy_diffs = []
    max_force_norm = []
    force_rms = []
    dft_res = []
    failed = 0
    freq_cnt = np.zeros(20)

    for i in range(len(true_pos_list)):
        if success_list is not None and not success_list[i]:
            failed += 1
        if dft_res_list[i][1] is None:
            continue
        batch = torch.zeros((true_pos_list[i].shape[0]), dtype=torch.long, device='cpu')
        rmsd = rmsd_loss(torch.tensor(pred_pos_list[i]).cpu(), torch.tensor(true_pos_list[i]).cpu(), batch)
        dmae = d_mae_loss(torch.tensor(pred_pos_list[i]).cpu(), torch.tensor(true_pos_list[i]).cpu(), batch)
        rmsds.append(rmsd.item())
        dmaes.append(dmae.item())
        energy_diffs.append(energy_diff_list[i].item())
        max_force_norm.append(np.max(np.linalg.norm(dft_res_list[i][1], ord=2, axis=-1)).item())
        force_rms.append(np.sqrt(np.mean(np.square(np.linalg.norm(dft_res_list[i][1], ord=2, axis=-1)))).item())
        res = count_negative_eig(dft_res_list[i][-1]['freq_wavenumber'])
        freq_cnt[res] += 1
        dft_res.append(dft_res_list[i])

    return rmsds, dmaes, energy_diffs, dft_res#, max_force_norm, force_rms

In [ ]:
def reaction_stats_csv(true_pos_list, pred_pos_list, energy_diff_list, dft_res_list, success_list=None):
    rmsds = []
    dmaes = []
    energy_diffs = []
    max_force_norm = []
    force_rms = []
    freq_res = []
    failed = 0

    for i in range(len(true_pos_list)):
        if success_list is not None and not success_list[i]:
            failed += 1
        if dft_res_list[i][1] is None:
            continue
        batch = torch.zeros((true_pos_list[i].shape[0]), dtype=torch.long, device='cpu')
        rmsd = rmsd_loss(torch.tensor(pred_pos_list[i]).cpu(), torch.tensor(true_pos_list[i]).cpu(), batch)
        dmae = d_mae_loss(torch.tensor(pred_pos_list[i]).cpu(), torch.tensor(true_pos_list[i]).cpu(), batch)
        rmsds.append(rmsd.item())
        dmaes.append(dmae.item())
        energy_diffs.append(energy_diff_list[i].item())
        max_force_norm.append(np.max(np.linalg.norm(dft_res_list[i][1], ord=2, axis=-1)).item())
        force_rms.append(np.sqrt(np.mean(np.square(np.linalg.norm(dft_res_list[i][1], ord=2, axis=-1)))).item())
        freq_res_i = count_negative_eig(dft_res_list[i][-1]['freq_wavenumber'])
        freq_res.append(freq_res_i)

    return {
        'rmsd': rmsds,
        'dmae': dmaes,
        'energy_diff': energy_diffs,
        'max_force_norm': max_force_norm,
        'force_rms': force_rms,
        'neg_freq_cnt': freq_res
    }

Baseline NeuralNEB

In [ ]:
with open('res_neuralneb.pickle', 'rb') as f:
    temp = pickle.load(f)
success_neb = temp['neb']['success']
dft_res_neb = temp['neb']['dft_res']
pred_trans_pos_neb = [get_positions_from_pyscf(temp_res) for temp_res in dft_res_neb]
energy_diff_neb = temp['neb']['barriers_dft']
true_trans_pos = [data.transition_state_pos for data in dataloader['test']]

In [ ]:
res_neuralNEB = reaction_stats(true_trans_pos, pred_trans_pos_neb, energy_diff_neb, dft_res_neb, success_neb)

In [ ]:
res_neuralNEB_csv = reaction_stats_csv(true_trans_pos, pred_trans_pos_neb, energy_diff_neb, dft_res_neb, success_neb)

Baselines learnTS (PSI-based)

In [ ]:
with open('energy_res_learnts.pickle', 'rb') as f:
    temp = pickle.load(f)
dft_res_model = temp['dft_res']
energy_diff_model = temp['energy_diffs']

In [ ]:
with open('res_learnts.pickle', 'rb') as f:
    temp = pickle.load(f)
pred_trans_pos_model = temp['pred_transition_state_pos']
true_trans_pos = temp['true_transition_state_pos']
true_trans_energy = temp['true_trans_energy']

In [ ]:
pred_trans_energy_neb = [temp_res[0].e_tot * AU2EV for temp_res in dft_res_neb]
energy_diff_neb = [abs(temp1 - temp2) for (temp1, temp2) in zip(true_trans_energy, pred_trans_energy_neb)]

In [ ]:
res_learnTS = reaction_stats(true_trans_pos, pred_trans_pos_model, energy_diff_model, dft_res_model)

In [ ]:
res_learnTS_csv = reaction_stats_csv(true_trans_pos, pred_trans_pos_model, energy_diff_model, dft_res_model)

Baseline OA-Reactdiff

In [ ]:
with open('res_reactdiff.pickle', 'rb') as f:
    temp = pickle.load(f)
pred_trans_pos_model = temp['pred_transition_state_pos']
true_trans_pos = temp['true_transition_state_pos']
dft_res_model = temp['dft_res_orig']
energy_diff_model = temp['energy_diff_orig']

In [ ]:
res_reactdiff = reaction_stats(true_trans_pos[:-1], pred_trans_pos_model, energy_diff_model, dft_res_model)

In [ ]:
res_reactdiff_csv = reaction_stats_csv(true_trans_pos[:-1], pred_trans_pos_model, energy_diff_model, dft_res_model)

Baseline react-ot

In [ ]:
with open('res_reactot.pickle', 'rb') as f:
    temp = pickle.load(f)
pred_trans_pos_model = temp['pred_transition_state_pos']
true_trans_pos = temp['true_transition_state_pos']
dft_res_model = temp['dft_res_orig']
energy_diff_model = temp['energy_diff_orig']

In [ ]:
res_reactot = reaction_stats(true_trans_pos, pred_trans_pos_model, energy_diff_model, dft_res_model)

In [ ]:
res_reactot_csv = reaction_stats_csv(true_trans_pos, pred_trans_pos_model, energy_diff_model, dft_res_model)

Our Method

In [ ]:
with open('res_fm.pickle', 'rb') as f:
    temp = pickle.load(f)
dft_res_model = temp['fm']['dft_res']
energy_diff_model = temp['fm']['barriers_dft']
pred_trans_pos_model = temp['fm']['pred_trans_pos']
true_trans_pos = [data.transition_state_pos for data in dataloader['test']]

In [ ]:
res_ours = reaction_stats(true_trans_pos, pred_trans_pos_model, energy_diff_model, dft_res_model)

In [ ]:
res_ours_csv = reaction_stats_csv(true_trans_pos, pred_trans_pos_model, energy_diff_model, dft_res_model)

In [ ]:
matplotlib.rcParams.update({'font.size': 8})
matplotlib.rcParams.update({'axes.titlesize': 6})
matplotlib.rcParams.update({'axes.labelsize': 6})
matplotlib.rcParams.update({'legend.fontsize': 6})
matplotlib.rcParams.update({'xtick.labelsize': 6})
matplotlib.rcParams.update({'ytick.labelsize': 6})
matplotlib.rcParams.update({'axes.linewidth': 1.0, 'xtick.major.width': 0.8, 'ytick.major.width': 0.8, 'xtick.minor.width': 0.6, 'ytick.minor.width': 0.6, 'grid.linewidth': 0.5})
matplotlib.rcParams.update({'font.family': 'sans-serif', 'font.sans-serif': 'Arial'})
# matplotlib.rcParams.update({'font.family': 'sans-serif', 'font.sans-serif': 'Times New Roman'})

In [ ]:
width = 180 / 25.4
height = 100 / 25.4
figs = plt.figure(figsize=(width, height))
subfigs = figs.subfigures(1, 3, width_ratios=[1, 1, 1], wspace=0.15, hspace=0.0)

In [ ]:
data_force = pd.DataFrame([
    pd.Series([np.max(np.linalg.norm(temp[1], ord=2, axis=-1)).item() for temp in res_neuralNEB[-1]], name = 'NeuralNEB'),
    pd.Series([np.max(np.linalg.norm(temp[1], ord=2, axis=-1)).item() for temp in res_learnTS[-1]], name = 'PSI-based'),
    pd.Series([np.max(np.linalg.norm(temp[1], ord=2, axis=-1)).item() for temp in res_reactdiff[-1]], name = 'OA-ReactDiff + Best'),
    pd.Series([np.max(np.linalg.norm(temp[1], ord=2, axis=-1)).item() for temp in res_reactot[-1]], name = 'ReactOT'),
    pd.Series([np.max(np.linalg.norm(temp[1], ord=2, axis=-1)).item() for temp in res_ours[-1]], name = 'TS-DFM')
]).transpose()

data_hessian = pd.DataFrame([
    pd.Series([count_negative_eig(temp[-1]['freq_wavenumber'])==1 for temp in res_neuralNEB[-1]], name = 'NeuralNEB'),
    pd.Series([count_negative_eig(temp[-1]['freq_wavenumber'])==1 for temp in res_learnTS[-1]], name = 'PSI-based'),
    pd.Series([count_negative_eig(temp[-1]['freq_wavenumber'])==1 for temp in res_reactdiff[-1]], name = 'OA-ReactDiff + Best'),
    pd.Series([count_negative_eig(temp[-1]['freq_wavenumber'])==1 for temp in res_reactot[-1]], name = 'ReactOT'),
    pd.Series([count_negative_eig(temp[-1]['freq_wavenumber'])==1 for temp in res_ours[-1]], name = 'TS-DFM')
]).transpose()

In [ ]:
data_force[data_force > 10.0] = 10.0

In [ ]:
fig = subfigs[0]

fig.text(-0.1, 0.95, 'a', ha='center', va='center', fontsize=10)

ax_box, ax_ecdf = fig.subplots(nrows=2, sharex=True, gridspec_kw={'height_ratios': [1, 1.2], 'hspace': 0.0})

data = pd.DataFrame([
    pd.Series(res_neuralNEB[0], name = 'NeuralNEB'),
    pd.Series(res_learnTS[0], name = 'PSI-based'),
    pd.Series(res_reactdiff[0], name = 'OA-ReactDiff'),
    pd.Series(res_reactot[0], name = 'React-OT'),
    pd.Series(res_ours[0], name = 'TS-DFM')
]).transpose()

sns.boxplot(data=data, ax=ax_box, palette='Set2', orient='h', whis=10.0, boxprops=dict(alpha=0.7), width=0.9)

sns.ecdfplot(data=data, ax=ax_ecdf, palette='Set2', linewidth=2.0, legend=True)

ax_box.tick_params(axis='both', which='both', bottom=False, top=False, left=False, right=False)

for i in range(len(data.columns)):
    values = data.iloc[:,i].values
    sizes = data_force.iloc[:,i].values * 4
    flag = data_hessian.iloc[:,i].values
    
    y_pos = np.random.uniform(-0.2, -0.1, size=len(values)) + i
    
    c = [temp + 0.2 if temp < 0.8 else 1.0 for temp in sns.color_palette('Set2')[i]]
    ax_box.scatter(
        values[flag==True], 
        y_pos[flag==True],
        s=sizes[flag==True],
        # alpha=0.8, 
        edgecolor='gray',
        color=sns.color_palette('Set2')[i],  
        linewidth=0.5,
        zorder=3, 
    )

    y_pos = np.random.uniform(0.1, 0.2, size=len(values)) + i

    c = [temp + 0.4 if temp < 0.6 else 1.0 for temp in sns.color_palette('Set2')[i]]
    ax_box.scatter(
        values[flag==False], 
        y_pos[flag==False],
        s=sizes[flag==False],
        # alpha=0.5, 
        edgecolor='gray',
        color=c,  
        linewidth=0.5,
        zorder=3, 
    )

ax_box.set_yticks([])

ax_ecdf.set_xlabel(r'RMSD (Å)')
ax_ecdf.set_xscale('log')
ax_ecdf.set_xlim([1e-2, 2.0])
ax_ecdf.set_xticks([1e-2, 3.16e-2, 1e-1, 3.16e-1, 1e0, 2e0])
ax_ecdf.set_xticklabels([1e-2, '', 1e-1, '', 1e0, 2e0])
ax_ecdf.grid(True, linestyle='--', alpha=0.7)  

In [ ]:
fig = subfigs[1]

fig.text(-0.1, 0.95, 'b', ha='center', va='center', fontsize=10)

ax_box, ax_ecdf = fig.subplots(nrows=2, sharex=True, gridspec_kw={'height_ratios': [1, 1.2], 'hspace': 0.0})

data = pd.DataFrame([
    pd.Series(res_neuralNEB[1], name = 'NeuralNEB'),
    pd.Series(res_learnTS[1], name = 'PSI-based'),
    pd.Series(res_reactdiff[1], name = 'OA-ReactDiff'),
    pd.Series(res_reactot[1], name = 'React-OT'),
    pd.Series(res_ours[1], name = 'TS-DFM')
]).transpose()

sns.boxplot(data=data, ax=ax_box, palette='Set2', orient='h', whis=10.0, boxprops=dict(alpha=0.7), width=0.9)

sns.ecdfplot(data=data, ax=ax_ecdf, palette='Set2', linewidth=2.0, legend=True)

ax_box.tick_params(axis='both', which='both', bottom=False, top=False, left=False, right=False)

for i in range(len(data.columns)):
    values = data.iloc[:,i].values
    sizes = data_force.iloc[:,i].values * 4
    flag = data_hessian.iloc[:,i].values
    
    y_pos = np.random.uniform(-0.2, -0.1, size=len(values)) + i
    
    c = [temp + 0.2 if temp < 0.8 else 1.0 for temp in sns.color_palette('Set2')[i]]
    ax_box.scatter(
        values[flag==True], 
        y_pos[flag==True],
        s=sizes[flag==True],
        # alpha=0.8, 
        edgecolor='gray',
        color=sns.color_palette('Set2')[i],  
        linewidth=0.5,
        zorder=3, 
    )

    y_pos = np.random.uniform(0.1, 0.2, size=len(values)) + i

    c = [temp + 0.4 if temp < 0.6 else 1.0 for temp in sns.color_palette('Set2')[i]]
    ax_box.scatter(
        values[flag==False], 
        y_pos[flag==False],
        s=sizes[flag==False],
        # alpha=0.5, 
        edgecolor='gray',
        color=c,  
        linewidth=0.5,
        zorder=3, 
    )

ax_box.set_yticks([])

ax_ecdf.set_xlabel(r'DMAE (Å)')
ax_ecdf.set_xscale('log')
ax_ecdf.set_xlim([5e-3, 1e0])
ax_ecdf.set_xticks([5e-3, 1e-2, 3.16e-2, 1e-1, 3.16e-1, 1e0])
ax_ecdf.set_xticklabels([5e-3, 1e-2, '', 1e-1, '', 1e0])
ax_ecdf.grid(True, linestyle='--', alpha=0.7)  

In [ ]:
fig = subfigs[2]

fig.text(-0.1, 0.95, 'c', ha='center', va='center', fontsize=10)

ax_box, ax_ecdf = fig.subplots(nrows=2, sharex=True, gridspec_kw={'height_ratios': [1, 1.2], 'hspace': 0.0})

data = pd.DataFrame([
    pd.Series(res_neuralNEB[2], name = 'NeuralNEB'),
    pd.Series(res_learnTS[2], name = 'PSI-based'),
    pd.Series(res_reactdiff[2], name = 'OA-ReactDiff'),
    pd.Series(res_reactot[2], name = 'React-OT'),
    pd.Series(res_ours[2], name = 'TS-DFM')
]).transpose()

sns.boxplot(data=data, ax=ax_box, palette='Set2', orient='h', whis=30.0, boxprops=dict(alpha=0.7), width=0.9)

sns.ecdfplot(data=data, ax=ax_ecdf, palette='Set2', linewidth=2.0, legend=True)

ax_box.tick_params(axis='both', which='both', bottom=False, top=False, left=False, right=False)

for i in range(len(data.columns)):
    values = data.iloc[:,i].values
    sizes = data_force.iloc[:,i].values * 4
    flag = data_hessian.iloc[:,i].values
    
    y_pos = np.random.uniform(-0.2, -0.1, size=len(values)) + i
    
    c = [temp + 0.2 if temp < 0.8 else 1.0 for temp in sns.color_palette('Set2')[i]]
    ax_box.scatter(
        values[flag==True], 
        y_pos[flag==True],
        s=sizes[flag==True],
        # alpha=0.8, 
        edgecolor='gray',
        color=sns.color_palette('Set2')[i],  
        linewidth=0.5,
        zorder=3, 
    )

    y_pos = np.random.uniform(0.1, 0.2, size=len(values)) + i

    c = [temp + 0.4 if temp < 0.6 else 1.0 for temp in sns.color_palette('Set2')[i]]
    ax_box.scatter(
        values[flag==False], 
        y_pos[flag==False],
        s=sizes[flag==False],
        # alpha=0.5, 
        edgecolor='gray',
        color=c,  
        linewidth=0.5,
        zorder=3, 
    )

ax_box.set_yticks([])

ax_ecdf.set_xlabel('$|\Delta E_{\\text{TS}}|$ (eV)')
ax_ecdf.set_xscale('log')
ax_ecdf.set_xlim([2e-3, 5e0])
ax_ecdf.set_xticks([2e-3, 1e-2, 3.16e-2, 1e-1, 3.16e-1, 1e0, 5e0])
ax_ecdf.set_xticklabels([2e-3, 1e-2, '', 1e-1, '', 1e0, 5e0])
ax_ecdf.grid(True, linestyle='--', alpha=0.7)

In [ ]:
figs.tight_layout()
figs.savefig('res_acc.eps', bbox_inches='tight', dpi=1200)
plt.show()

In [ ]:
import pandas as pd

neuralNEB_df = pd.DataFrame(res_neuralNEB_csv)
psi_based_df = pd.DataFrame(res_learnTS_csv)
reactdiff_df = pd.DataFrame(res_reactdiff_csv)
reactot_df = pd.DataFrame(res_reactot_csv)
ours_df = pd.DataFrame(res_ours_csv)

with pd.ExcelWriter('res_acc.xlsx') as writer:
    neuralNEB_df.to_excel(writer, sheet_name='NeuralNEB', index=False)
    psi_based_df.to_excel(writer, sheet_name='PSI_Based', index=False)
    reactdiff_df.to_excel(writer, sheet_name='ReactDiff', index=False)
    reactot_df.to_excel(writer, sheet_name='ReactOT', index=False)
    ours_df.to_excel(writer, sheet_name='TS-DFM', index=False)